# 02 - Preprocessing Pipeline

Notebook này tạo dữ liệu sạch + artifacts cho training và backend inference.

## SECTION 1: Load & Select Columns

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

warnings.filterwarnings('ignore')

# Robust dataset path detection
candidate_paths = [
    Path('dataset/DataCoSupplyChainDataset.csv'),
    Path('../dataset/DataCoSupplyChainDataset.csv'),
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Cannot find dataset/DataCoSupplyChainDataset.csv')

PROJECT_ROOT = DATA_PATH.resolve().parents[1]
BACKEND_ML_DIR = PROJECT_ROOT / 'backend' / 'ml'
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
BACKEND_ML_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Using dataset path: {DATA_PATH.resolve()}')
print(f'Artifacts dir: {BACKEND_ML_DIR}')
print(f'Processed dir: {PROCESSED_DIR}')

df = pd.read_csv(DATA_PATH, encoding='cp1252')

FEATURE_COLS = [
    'Shipping Mode',
    'Days for shipment (scheduled)',
    'Market',
    'Order Region',
    'Category Name',
    'Customer Segment',
    'order date (DateOrders)',
    'Benefit per order',
    'Order Item Discount Rate',
    'Department Name',
]
TARGET = 'Late_delivery_risk'

required_cols = FEATURE_COLS + [TARGET]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

df_model = df[required_cols].copy()
print('Initial shape:', df_model.shape)
print('Columns selected:', df_model.columns.tolist())

## SECTION 2: Feature Engineering

In [ ]:
df_model['order date (DateOrders)'] = pd.to_datetime(
    df_model['order date (DateOrders)'],
    errors='coerce'
)

df_model['Month'] = df_model['order date (DateOrders)'].dt.month
df_model['Day_of_Week'] = df_model['order date (DateOrders)'].dt.dayofweek
df_model['Is_Peak_Season'] = df_model['Month'].isin([10, 11, 12]).astype(int)
df_model['Quarter'] = df_model['order date (DateOrders)'].dt.quarter

df_model.drop(columns=['order date (DateOrders)'], inplace=True)

print('Shape after feature engineering:', df_model.shape)
print('Added columns: Month, Day_of_Week, Is_Peak_Season, Quarter')

## SECTION 3: Handle Missing Values

In [ ]:
print('Missing count BEFORE:')
print(df_model.isnull().sum().sort_values(ascending=False))

num_cols_all = df_model.select_dtypes(include='number').columns.tolist()
num_cols_features_only = [c for c in num_cols_all if c != TARGET]
cat_cols_all = df_model.select_dtypes(include='object').columns.tolist()

for col in num_cols_features_only:
    df_model[col] = df_model[col].fillna(df_model[col].median())

for col in cat_cols_all:
    df_model[col] = df_model[col].fillna('Unknown')

print()
print('Missing count AFTER:')
print(df_model.isnull().sum().sort_values(ascending=False))
print()
print('Total missing after fill:', int(df_model.isnull().sum().sum()))

## SECTION 4: Encode Categorical Features

In [ ]:
CAT_COLS = [
    'Shipping Mode',
    'Market',
    'Order Region',
    'Category Name',
    'Customer Segment',
    'Department Name',
]

encoding_map = {}

for col in CAT_COLS:
    le = LabelEncoder()
    df_model[f'{col}_enc'] = le.fit_transform(df_model[col].astype(str))
    encoding_map[col] = {str(label): int(idx) for idx, label in enumerate(le.classes_)}

# Export encoding map
encoding_map_path = BACKEND_ML_DIR / 'encoding_map.json'
with open(encoding_map_path, 'w', encoding='utf-8') as f:
    json.dump(encoding_map, f, indent=2, ensure_ascii=False)

# Keep only encoded versions for categorical columns
for col in CAT_COLS:
    df_model.drop(columns=[col], inplace=True)

print(f'✅ Saved encoding map: {encoding_map_path}')
print('Encoding map keys:', list(encoding_map.keys()))

## SECTION 5: Define Final Feature List

In [ ]:
FINAL_FEATURES = [
    'Shipping Mode_enc',
    'Days for shipment (scheduled)',
    'Market_enc',
    'Order Region_enc',
    'Category Name_enc',
    'Customer Segment_enc',
    'Department Name_enc',
    'Month',
    'Day_of_Week',
    'Is_Peak_Season',
    'Quarter',
    'Benefit per order',
    'Order Item Discount Rate',
]

missing_final = [c for c in FINAL_FEATURES if c not in df_model.columns]
if missing_final:
    raise ValueError(f'Missing FINAL_FEATURES columns: {missing_final}')

feature_names_path = BACKEND_ML_DIR / 'feature_names.json'
with open(feature_names_path, 'w', encoding='utf-8') as f:
    json.dump(FINAL_FEATURES, f, indent=2, ensure_ascii=False)

X = df_model[FINAL_FEATURES].copy()
y = df_model[TARGET].copy()

print(f'✅ Saved feature names: {feature_names_path}')
print('Total features:', len(FINAL_FEATURES))

## SECTION 6: Train/Val/Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print('Split sizes:')
print('X_train:', X_train.shape)
print('X_val  :', X_val.shape)
print('X_test :', X_test.shape)

## SECTION 7: Scale Numerical Features

In [ ]:
NUM_COLS = [
    'Days for shipment (scheduled)',
    'Benefit per order',
    'Order Item Discount Rate',
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_val_scaled[NUM_COLS] = scaler.transform(X_val[NUM_COLS])
X_test_scaled[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

scaler_path = BACKEND_ML_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'✅ Saved scaler: {scaler_path}')

## SECTION 8: Verify & Save

In [ ]:
# Print shapes
print('Shapes after preprocessing:')
print('X_train:', X_train_scaled.shape, '| y_train:', y_train.shape)
print('X_val  :', X_val_scaled.shape, '| y_val  :', y_val.shape)
print('X_test :', X_test_scaled.shape, '| y_test :', y_test.shape)

# Class distributions
print()
print('Class distribution (train):')
print(y_train.value_counts(normalize=True))
print()
print('Class distribution (val):')
print(y_val.value_counts(normalize=True))
print()
print('Class distribution (test):')
print(y_test.value_counts(normalize=True))

# Save processed datasets for training notebook
X_train_scaled.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_val_scaled.to_csv(PROCESSED_DIR / 'X_val.csv', index=False)
X_test_scaled.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)

y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_val.to_csv(PROCESSED_DIR / 'y_val.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

print()
print('✅ Processed datasets saved to:', PROCESSED_DIR)

# Verify artifacts are loadable
with open(BACKEND_ML_DIR / 'encoding_map.json', 'r', encoding='utf-8') as f:
    loaded_encoding_map = json.load(f)

loaded_scaler = joblib.load(BACKEND_ML_DIR / 'scaler.pkl')

print('✅ Verified encoding_map.json loaded. Keys:', list(loaded_encoding_map.keys()))
print('✅ Verified scaler.pkl loaded. Type:', type(loaded_scaler).__name__)
print('✅ Preprocessing complete. Artifacts saved to backend/ml/')